# Agentic Portfolio Construction — Full Pipeline Demo
**Fordham MSQF Capstone 2026**

Five AI agents working in sequence to build a personalized portfolio recommendation:

```
Research Agent  →  Profile Agent  →  Allocation Agent
                                           ↕ (FLAG loop, max 3x)
                                       Risk Agent
                                           ↓
                                    Compliance Agent
                                           ↓
                                     AdvisorPackage
```

**Every number is deterministic** (pandas, numpy, FRED data, closed-form formulas).  
The LLM only writes rationale text and reasoning traces — it never invents a number.

---
### Prerequisites
1. Install deps: `pip install -r requirements.txt`
2. **First run only** — populate the data cache: run Section 1 below (~10 min, requires WRDS login)
3. Set env vars (optional — pipelines fall back gracefully without them):
   - `FRED_API_KEY` — free at fred.stlouisfed.org; uses 4.4% DGS10 fallback if missing
   - `ANTHROPIC_API_KEY` — Claude API; LLM rationale cells are skipped if missing

---
## 0. Environment Setup
Run this first every session.

In [ ]:
import os, sys, importlib, logging

# Ensure project root is on sys.path regardless of launch directory
PROJECT_ROOT = os.path.dirname(os.path.abspath('.'))
if os.path.basename(os.getcwd()) != os.path.basename(PROJECT_ROOT):
    PROJECT_ROOT = os.getcwd()  # already at root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

importlib.invalidate_caches()
logging.basicConfig(level=logging.WARNING)  # suppress INFO noise; flip to INFO to trace loops

FRED_KEY      = os.environ.get('FRED_API_KEY')
ANTHROPIC_KEY = os.environ.get('ANTHROPIC_API_KEY')
print(f'FRED_API_KEY set:      {bool(FRED_KEY)}')
print(f'ANTHROPIC_API_KEY set: {bool(ANTHROPIC_KEY)}')

---
## 1. One-Time Data Cache Setup

The pipeline reads from `data/storage/` (gitignored, ~500 MB).  
Run the cells below **once** to populate it, then skip this section on future runs.

| File | Source | Used by |
|---|---|---|
| `crsp_monthly.parquet` | WRDS/CRSP | Allocation (Black-Litterman market weights) |
| `crsp_daily.parquet` | WRDS/CRSP | Risk (VaR, drawdown, factor exposures) |
| `ff_risk_factors.parquet` | WRDS/FF | Allocation + Risk (factor model) |
| `ff12_monthly.parquet` | Ken French | Research (regime feature validation) |
| `bls_oes.parquet` | BLS OES 2023 | Profile (salary distributions) |
| `fred_macro.parquet` | FRED | Research (13 macro series) |
| `permno_map.json` | WRDS | Risk (ticker → CRSP permno) |
| `mkt_cap_weights.json` | WRDS | Allocation (BL prior weights) |

In [ ]:
from pathlib import Path

REQUIRED = [
    'crsp_monthly.parquet', 'crsp_daily.parquet', 'ff_risk_factors.parquet',
    'fred_macro.parquet',   'bls_oes.parquet',    'ff12_monthly.parquet',
    'permno_map.json',      'mkt_cap_weights.json',
]
storage = Path('data/storage')

missing = [f for f in REQUIRED if not (storage / f).exists()]
if not missing:
    print('All data files present. Skip to Section 2.')
else:
    for f in REQUIRED:
        mark = 'OK     ' if (storage / f).exists() else 'MISSING'
        print(f'  {mark}  {f}')

In [ ]:
# WRDS connection — prompts for Fordham WRDS username + password on first run
from data.fetch.wrds import get_connection
conn = get_connection()
print('Connected to WRDS')

In [ ]:
from agents.allocation.adapters import DEFAULT_TICKERS
from data.fetch.wrds import fetch_crsp_monthly, fetch_crsp_daily, fetch_ff_factors

fetch_crsp_monthly(DEFAULT_TICKERS, conn=conn)   # crsp_monthly.parquet + permno_map + mkt_cap_weights
fetch_crsp_daily(DEFAULT_TICKERS, conn=conn)     # crsp_daily.parquet  (~5 min)
fetch_ff_factors(conn=conn)                      # ff_risk_factors.parquet
print('WRDS fetch done')

In [ ]:
from data.fetch.fred import fetch_fred_macro
from data.fetch.bls  import fetch_bls_oes
from data.fetch.factors import fetch_ff12

fetch_fred_macro(fred_api_key=FRED_KEY)  # fred_macro.parquet
fetch_bls_oes()                          # bls_oes.parquet
fetch_ff12()                             # ff12_monthly.parquet
print('Public data fetch done')

---
## 2. Agent 1 — Research Agent: Macro Regime Detection

**What it does:** Pulls 13 FRED macro series (yield curve, unemployment, CPI, credit spread, …),  
runs PELT change-point detection to find structural breaks, clusters the resulting segments with  
K-means, then trains an XGBoost classifier against five historically-labelled anchor windows.  
A 6-month rolling majority vote smooths out noise.

**Output:** `MacroRegimeSnapshot` — the current regime label, confidence, and key FRED signals.  
This snapshot is passed unchanged to every downstream agent so all five agents share the same  
macro view.

In [ ]:
from agents.research.research_agent import run_research_agent

# compare_models=False skips the HMM/GMM benchmarking (paper validation only)
macro = run_research_agent(fred_api_key=FRED_KEY, compare_models=False)

print(f'Regime:          {macro.regime_label}')
print(f'Confidence:      {macro.regime_confidence:.0%}')
print(f'Prior regime:    {macro.prior_regime}')
print(f'Regime change:   {macro.regime_change_detected}')
print(f'Low confidence:  {macro.is_low_confidence}')
print(f'As of:           {macro.as_of}')
print()
print('--- Key FRED signals ---')
print(f'Yield curve (10Y-2Y):  {macro.yield_curve:+.2f} pp')
print(f'Fed funds rate:         {macro.fed_funds:.2f}%')
print(f'Unemployment:           {macro.unemployment:.1f}%')
print(f'CPI YoY:                {macro.cpi:.1f}%')
print(f'Credit spread:          {macro.credit_spread:.2f} pp')

---
## 3. Agent 2 — Profile Agent: Human Capital Valuation

**What it does:** Maps BLS OES May 2023 salary data to 9 occupational personas, looks up  
income-equity beta (β) and correlation (ρ) from a calibrated table (Ibbotson et al. 2007),  
then computes four derived numbers that are the engine of portfolio differentiation:

| Formula | Meaning |
|---|---|
| `HC = Salary × [1 − (1+r)^{−n}] / r` | Present value of future earnings (annuity, FRED DGS10 discount) |
| `implicit_equity_exposure = hc_share × β` | Equity risk already carried through the career |
| `effective_risk_budget = (FC + HC×(1−σ)) / total_wealth` | Total risk capacity across the balance sheet |
| `portfolio_equity_target = risk_budget − implicit_equity_exposure` | Equity headroom left for the investment portfolio |

A tech exec with β=1.2 already has ~90% implicit equity exposure through their career and RSUs —  
their portfolio should be mostly bonds. A tenured professor with β=0.05 has near-zero implicit  
equity exposure and can hold a much higher equity allocation.

**Output:** `list[ProfileAgentOutput]` — 9 validated Pydantic objects, one per BLS occupation.

In [ ]:
from agents.profile.profile_agent import run_profile_agent

profiles = run_profile_agent(fred_api_key=FRED_KEY)
print(f'Built {len(profiles)} profiles')
print()

# Show the balance-sheet differentiation across all personas
import pandas as pd
summary = pd.DataFrame([{
    'Client':          p.client_id,
    'HC type':         p.human_capital_type.value,
    'β':               f'{p.income_equity_beta:.2f}',
    'Implicit eq exp': f'{p.implicit_equity_exposure:.1%}',
    'Risk budget':     f'{p.effective_risk_budget:.1%}',
    'HC % wealth':     f'{p.human_capital_pct_of_total:.0f}%',
} for p in profiles])
print(summary.to_string(index=False))

In [ ]:
# Pick one persona to run through the rest of the pipeline
# SOC 15-1252 = Software Developers (median salary, equity-like HC)
profile = next(p for p in profiles if p.client_id == 'bls_15-1252_p50')

print(f'Selected:               {profile.client_id}')
print(f'Career type:            {profile.career_type}')
print(f'Age / horizon:          {profile.age} yrs / {profile.investment_horizon_years} yrs')
print(f'Financial capital:      ${profile.financial_capital:>12,.0f}')
print(f'Human capital (PV):     ${profile.human_capital_valuation:>12,.0f}')
print(f'Total wealth:           ${profile.total_wealth:>12,.0f}')
print(f'HC type:                {profile.human_capital_type.value}')
print(f'Income beta (β):        {profile.income_equity_beta:.2f}')
print(f'Income-equity corr (ρ): {profile.income_equity_correlation:.2f}')
print(f'Income volatility (σ):  {profile.income_volatility_sigma:.2f}')
print(f'Implicit equity exp:    {profile.implicit_equity_exposure:.1%}')
print(f'Effective risk budget:  {profile.effective_risk_budget:.1%}')
print(f'Risk tolerance:         {profile.risk_tolerance_level.value}')
print(f'Employer sector:        {profile.industry_exposure_sector}')

---
## 4. Agents 3 + 4 — Allocation ↔ Risk Loop

**Allocation Agent:** Uses Black-Litterman to build a portfolio starting from CRSP market-cap  
equilibrium weights, tilts toward the macro regime view, then subtracts `implicit_equity_exposure`  
to keep total (career + portfolio) equity risk appropriate for the client. Calls Claude to write  
a per-ticker rationale citing the client's specific numbers.

**Risk Agent:** Runs deterministic stress tests (VaR 95/99, max drawdown, sector concentration,  
HC-correlation-adjusted limits) and returns `APPROVE`, `FLAG`, or `REJECT`.

**FLAG loop (max 3 iterations):** On FLAG, the violated constraints are fed back to the Allocation  
Agent which re-runs with tighter limits. REJECT after three FAGs is terminal — the portfolio is  
still delivered but the orchestrator records the failure in metadata.

The full orchestrator also runs a **Compliance → Allocation loop** (max 2 iterations) after this.

In [ ]:
# Enable INFO logging to see the FLAG loop iterations in real time
logging.getLogger().setLevel(logging.INFO)
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s', datefmt='%H:%M:%S', force=True)

from agents.orchestrator.orchestrator import run_pipeline

pkg = run_pipeline(profile, macro, fred_api_key=FRED_KEY)

logging.getLogger().setLevel(logging.WARNING)  # quiet again for result cells
print('\nPipeline complete.')

---
## 5. Results

In [ ]:
# Pipeline metadata — loop counts and final decisions
m = pkg.metadata
print('=== PIPELINE METADATA ===')
print(f'  Final risk decision:      {m.final_risk_decision.value}')
print(f'  Final compliance status:  {m.final_compliance_status.value}')
print(f'  Risk FLAG revisions:      {m.risk_revisions}')
print(f'  Compliance revisions:     {m.compliance_revisions}')
if m.pipeline_warnings:
    print('  Warnings:')
    for w in m.pipeline_warnings:
        print(f'    - {w}')

In [ ]:
# Portfolio weights
weights = pkg.allocation.proposed_portfolio
df_w = pd.DataFrame(
    [{'Ticker': t, 'Weight': w, 'Weight %': f'{w:.1%}'}
     for t, w in sorted(weights.items(), key=lambda x: x[1], reverse=True)
     if w > 0.001]
)
print('=== PORTFOLIO WEIGHTS ===')
print(df_w.to_string(index=False))
print(f'\nTotal: {sum(weights.values()):.4f}')

In [ ]:
# Risk Agent output
r = pkg.risk
print('=== RISK AGENT OUTPUT ===')
print(f'Decision:   {r.risk_decision.value}')
if r.portfolio_volatility_annual is not None:
    print(f'Volatility: {r.portfolio_volatility_annual:.2%}  (annualized)')
print(f'Violations: {r.violations if r.violations else "None"}')

print('\nRegime stress tests (portfolio drawdown vs 80/20 benchmark):')
for regime, ev in r.regime_evaluation.items():
    status = 'PASS' if ev.passed else 'FAIL'
    print(f'  [{status}] {regime:<32}  portfolio {ev.portfolio_drawdown:.1%}  bench {ev.benchmark_drawdown:.1%}')

In [ ]:
# Compliance Agent output
c = pkg.compliance
print('=== COMPLIANCE AGENT OUTPUT ===')
print(f'Clearance:        {c.clearance}')
print(f'Status:           {c.compliance_status.value}')
print(f'Overall severity: {c.overall_severity.value}')
print(f'Recommendation:   {c.recommendation}')

if c.passed_checks:
    sample = ', '.join(c.passed_checks[:5])
    suffix = '...' if len(c.passed_checks) > 5 else ''
    print(f'\nPassed checks ({len(c.passed_checks)}): {sample}{suffix}')

if c.violations:
    print(f'\nViolations ({len(c.violations)}):')
    for v in c.violations:
        print(f'  [{v.severity.value}] {v.check}: {v.description}')

In [ ]:
# LLM-generated allocation rationale (one ticker as sample)
rationale = pkg.allocation.allocation_rationale
if rationale:
    sample_ticker = next(iter(rationale))
    print(f'=== ALLOCATION RATIONALE — {sample_ticker} ===')
    print(rationale[sample_ticker])
else:
    print('No rationale (ANTHROPIC_API_KEY not set)')

---
## 6. Batch Run — All 9 BLS Personas

Run `run_all()` to produce one `AdvisorPackage` per BLS occupation persona in a single call.  
The Research Agent runs once; the Profile → Allocation → Risk → Compliance chain runs 9 times in sequence.

In [ ]:
from agents.orchestrator.orchestrator import run_all

# Pass the already-computed macro snapshot to skip the Research Agent re-run
packages = run_all(fred_api_key=FRED_KEY, macro=macro)

print(f'\n{len(packages)} AdvisorPackages produced')
print()

# Summary table across all personas
rows = []
for p in packages:
    rows.append({
        'Client':       p.profile.client_id,
        'HC type':      p.profile.human_capital_type.value,
        'Risk':         p.metadata.final_risk_decision.value,
        'Compliance':   p.metadata.final_compliance_status.value,
        'FLAG revs':    p.metadata.risk_revisions,
        'Top holding':  max(p.allocation.proposed_portfolio, key=p.allocation.proposed_portfolio.get),
    })

print(pd.DataFrame(rows).to_string(index=False))